In [1]:
from pathlib import Path
import pandas as pd

project_dir = Path(r"C:\Users\HP-ZBOOK i7\graphrag_test\pilot_01")
csv_path = project_dir / "inspection" / "10_possible_duplicates.csv"

duplicates = pd.read_csv(csv_path)

print("Number of duplicate candidates:", len(duplicates))
print("Columns:", duplicates.columns.tolist())

duplicates.head(30)

Number of duplicate candidates: 2
Columns: ['title', 'normalized_title', 'type', 'description', 'frequency', 'degree']


,title,normalized_title,type,description,frequency,degree
0,"FTI-SCHWERPUNKT ""ENERGIEWENDE""",FTI SCHWERPUNKT ENERGIEWENDE,EVENT,"The FTI-Schwerpunkt ""Energiewende"" is a nation...",1,1
1,FTI-SCHWERPUNKT ENERGIEWENDE,FTI SCHWERPUNKT ENERGIEWENDE,EVENT,"The FTI-Schwerpunkt ""Energiewende"" is a resear...",1,2


In [2]:
entities_path = project_dir / "inspection" / "01_entities_all.csv"
entities = pd.read_csv(entities_path)

alias_terms = [
    "BMK",
    "BUNDESMINISTERIUM",
    "PHOTOVOLTAIK",
    "PV-",
    "NETZINFRASTRUKTURPLAN",
    "HACKNER",
    "ENERGIEWENDE",
]

pattern = "|".join(alias_terms)

alias_candidates = entities[
    entities["title"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True,
    )
][
    ["title", "type", "frequency", "degree"]
].sort_values("title")

alias_candidates

,title,type,frequency,degree
45,AGRI-PV-ANLAGEN,ORGANIZATION,4,5
62,BETREIBER:INNEN DER PV-ANLAGEN,PERSON,1,1
10,BMK,ORGANIZATION,7,16
178,BMK-AUSBILDUNGSINITIATIVE “JUST TRANSITION”,ORGANIZATION,1,3
311,BMK.GV.AT,ORGANIZATION,1,2
284,BUNDESMINISTERIUM FÜR ENERGIE,ORGANIZATION,1,0
286,BUNDESMINISTERIUM FÜR INNOVATION UND TECHNOLOGIE,ORGANIZATION,1,0
1,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,2,9
272,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,1,4
285,BUNDESMINISTERIUM FÜR MOBILITÄT,ORGANIZATION,1,0


## Stage 5 — Entity Resolution and Duplicate Audit

### Objective

The purpose of this stage was to determine whether GraphRAG represented the same real-world concept through multiple entity nodes.

The audit considered:

* Abbreviations
* Full names
* Shortened names
* Singular and plural forms
* Punctuation variants
* Reordered personal names
* Spelling variants
* Inconsistent entity types
* Related but genuinely distinct concepts

Entity resolution is different from entity extraction and entity typing.

A concept can be correctly extracted and correctly typed but still remain duplicated under several names.

---

## Automatic duplicate-candidate output

The existing file `10_possible_duplicates.csv` contained only **two rows**, representing one duplicate group:

```text
FTI-SCHWERPUNKT "ENERGIEWENDE"
FTI-SCHWERPUNKT ENERGIEWENDE
```

The two labels differ only through quotation marks.

They share the same normalized form:

```text
FTI SCHWERPUNKT ENERGIEWENDE
```

This is a clear punctuation-level duplicate and should be represented as one canonical entity.

However, the automatic file did not identify the more important semantic alias families found during manual inspection.

The file therefore detects only a narrow category of duplicates with nearly identical normalized titles.

---

## Important missed alias families

### Austrian ministry

GraphRAG created:

```text
BMK
```

and:

```text
BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE,
MOBILITÄT, INNOVATION UND TECHNOLOGIE
```

These refer to the same organization.

Recommended representation:

```text
Canonical label:
Bundesministerium für Klimaschutz, Umwelt, Energie,
Mobilität, Innovation und Technologie

Alias:
BMK
```

The output also contains fragments such as:

```text
BUNDESMINISTERIUM FÜR ENERGIE
BUNDESMINISTERIUM FÜR MOBILITÄT
BUNDESMINISTERIUM FÜR UMWELT
BUNDESMINISTERIUM FÜR INNOVATION UND TECHNOLOGIE
```

In this document, these may originate from portions of the ministry’s complete name rather than from separate ministries. They require source verification before being retained as independent organizations.

Contact elements such as:

```text
BMK.GV.AT
SERVICEBUERO@BMK.GV.AT
```

are not aliases for the ministry entity. They are a website and an email address. They should normally be stored as attributes rather than independent domain nodes.

---

### Austrian Photovoltaic Strategy

GraphRAG produced several variants:

```text
ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
PHOTOVOLTAIK-STRATEGIE
PV-STRATEGIE
```

These appear to refer to the same policy document.

Recommended representation:

```text
Canonical label:
Österreichische Photovoltaik-Strategie

Aliases:
Photovoltaik-Strategie
PV-Strategie
```

The strategy should receive one stable identifier and one controlled type such as `POLICY` or `STRATEGY`.

---

### Photovoltaics

GraphRAG produced forms including:

```text
PHOTOVOLTAIK
PHOTOVOLTAIK (PV)
PV
```

These should generally map to one technology concept:

```text
Canonical label:
Photovoltaics

German label:
Photovoltaik

Abbreviation:
PV
```

This is especially important for future multilingual processing.

---

### Photovoltaic installations

GraphRAG produced:

```text
PV-ANLAGE
PV-ANLAGEN
PHOTOVOLTAIKANLAGEN
```

These are singular, plural, and expanded-language variants.

Recommended representation:

```text
Canonical concept:
Photovoltaic Installation

Aliases:
PV-Anlage
PV-Anlagen
Photovoltaikanlage
Photovoltaikanlagen
```

The controlled KG may need to distinguish:

* The general technology `Photovoltaics`
* The installation class `Photovoltaic Installation`
* Individual physical installations, if represented later

These concepts should not be merged indiscriminately.

---

### Austrian network infrastructure plan

GraphRAG generated multiple variants of the integrated Austrian network infrastructure plan. The variants also received inconsistent types, including `EVENT` and `ORGANIZATION`.

Relevant labels include forms using:

```text
NIP
ÖNIP
```

The source primarily uses the Austrian abbreviation `ÖNIP`.

Recommended representation:

```text
Canonical label:
Integrierter österreichischer Netzinfrastrukturplan

Preferred abbreviation:
ÖNIP
```

`NIP` may be retained as an observed source or extraction variant but should not remain a separate entity.

---

### Johannes Hackner

GraphRAG produced:

```text
HACKNER, JOHANNES
JOHANNES HACKNER
```

These refer to the same person.

The difference results from bibliography-style name order.

Recommended representation:

```text
Canonical label:
Johannes Hackner

Observed source form:
Hackner, Johannes
```

---

### FTI focus: energy transition

GraphRAG produced:

```text
FTI-SCHWERPUNKT "ENERGIEWENDE"
FTI-SCHWERPUNKT ENERGIEWENDE
```

This is the only duplicate family detected by the simple normalized-title output.

Recommended representation:

```text
Canonical label:
FTI-Schwerpunkt Energiewende
```

Quotation marks should not create a separate node.

---

## Type-conflict duplicates

Some duplicate families received different raw GraphRAG types.

For example:

```text
PV-ANLAGE → ORGANIZATION
PV-ANLAGEN → EVENT
PHOTOVOLTAIKANLAGEN → ORGANIZATION
```

This means entity resolution cannot be performed safely by grouping only entities with the same type.

The system must consider that aliases may be incorrectly typed before resolution.

The same issue occurs for variants of the Austrian network infrastructure plan, which were classified as both `EVENT` and `ORGANIZATION`.

Entity typing and entity resolution must therefore be corrected jointly.

---

## Related but distinct concepts

The broader keyword search also returned many PV-related entities that should not automatically be merged.

Examples include:

```text
PV-INDUSTRIE
PV-BRANCHE
PV-BRANCHENVERTRETUNG
PV-FORSCHUNG
PV-LEHRE
PV-WERTSCHÖPFUNGSKETTE
PV-DACHGÄRTEN
PV-BIODIVERSITÄTSANLAGEN
PV-FREIFLÄCHENANLAGEN
PV-SYSTEMUMFELD
```

These share the term `PV`, but they represent different concepts.

For example:

* `PV-INDUSTRIE` refers to an industrial sector.
* `PV-BRANCHENVERTRETUNG` refers to an industry-representation actor.
* `PV-FORSCHUNG` refers to research activity.
* `PV-DACHGÄRTEN` refers to a specific application.
* `PV-WERTSCHÖPFUNGSKETTE` refers to the value chain.

Automatically merging entities based only on shared words would destroy meaningful distinctions.

Similarly:

```text
PV-ANLAGE
AGRI-PV-ANLAGEN
PV-BIODIVERSITÄTSANLAGEN
PV-FREIFLÄCHENANLAGEN
```

are related but may represent a hierarchy:

```text
Photovoltaic Installation
├── Agri-PV Installation
├── Biodiversity PV Installation
└── Ground-Mounted PV Installation
```

They should be linked through controlled relationships or class hierarchies rather than collapsed into one node.

---

## Limitations of normalized-title matching

Simple title normalization can detect differences such as:

* Quotation marks
* Capitalization
* Some punctuation
* Extra whitespace

It usually cannot detect:

* Abbreviation versus full name
* Singular versus plural
* German versus English labels
* Reordered personal names
* Acronyms
* Semantically equivalent paraphrases
* Typing inconsistencies
* Closely related but distinct concepts

Therefore:

```text
10_possible_duplicates.csv
```

significantly underestimates the true entity-resolution problem.

Its result of two candidate rows must not be interpreted as evidence that the graph contains only one duplicate entity family.

---

## Likely causes

### Independent chunk extraction

The same concept can appear in multiple chunks under different surface forms.

### Lack of a canonical vocabulary

GraphRAG was not given a domain alias dictionary containing mappings such as:

```text
BMK → full ministry name
PV → Photovoltaics
ÖNIP → full plan name
```

### Generic extraction prompt

The extraction prompt requests named entities but does not provide domain-specific canonicalization rules.

### Singular and plural variation

German grammatical forms create variants such as:

```text
PV-Anlage
PV-Anlagen
Photovoltaikanlagen
```

### Bibliographic name formatting

References often use:

```text
Surname, Firstname
```

while body text may use:

```text
Firstname Surname
```

### Type instability

The same concept may be assigned different raw types, preventing simple type-constrained matching.

---

## Recommended controlled resolution approach

A later controlled KG transformation should contain an alias table such as:

| Observed label           | Canonical entity                                | Match type             |
| ------------------------ | ----------------------------------------------- | ---------------------- |
| `BMK`                    | Full ministry name                              | Abbreviation           |
| `PV`                     | Photovoltaics                                   | Abbreviation           |
| `Photovoltaik`           | Photovoltaics                                   | German label           |
| `PV-Strategie`           | Austrian Photovoltaic Strategy                  | Short name             |
| `Photovoltaik-Strategie` | Austrian Photovoltaic Strategy                  | Short name             |
| `PV-Anlagen`             | Photovoltaic Installation                       | Plural variant         |
| `Photovoltaikanlagen`    | Photovoltaic Installation                       | Expanded variant       |
| `NIP`                    | Integrated Austrian Network Infrastructure Plan | Extraction variant     |
| `ÖNIP`                   | Integrated Austrian Network Infrastructure Plan | Preferred abbreviation |
| `Hackner, Johannes`      | Johannes Hackner                                | Reordered name         |

Every canonical entity should have:

* Stable identifier
* Preferred label
* Alternative labels
* Controlled type
* Source provenance
* Language information
* Manual-review status where necessary

---

## Stage 5 conclusion

The simple normalized-title duplicate detector identified only one punctuation-level duplicate family. Manual and keyword-based inspection revealed several more important alias families involving abbreviations, shortened titles, singular/plural forms, and reordered names.

The most important unresolved families include:

* BMK and the full ministry name
* Photovoltaics and PV
* Multiple Austrian PV Strategy labels
* Multiple PV-installation labels
* NIP and ÖNIP variants
* Johannes Hackner name-order variants
* FTI energy-transition focus punctuation variants

At the same time, many entities sharing the term `PV` are related but genuinely distinct. Record linkage must therefore avoid both:

* **False negatives:** failing to merge true aliases
* **False positives:** incorrectly merging related but separate concepts

Record linkage is a major downstream requirement. Domain-specific entity types may reduce some inconsistency, but they will not solve abbreviation, language, grammatical, or naming variation by themselves.

The evidence supports a controlled entity-resolution step after GraphRAG extraction and before loading the final KG.
